# VMTF-Net - Google Colab (A100 / V100) + Google Drive

1. Runtime → Change runtime type → **A100 GPU** (or V100), High-RAM if available.
2. Google Drive layout used below (edit `DRIVE_ROOT` if different):
```
MyDrive/vmtf/
├── data.zip          # train.csv, val.csv, test.csv and images/ at the ROOT of the zip (one file: Drive is slow with many small files)
├── kb/               # optional: claim knowledge base index
└── results/colab_a100/<experiment>/   # checkpoints, history.csv, results.json (written here, survives disconnects)
```
3. If the session disconnects, simply re-run all cells: the training cell resumes from `last_checkpoint.pt` on Drive.

⚠️ Keep every model of the same comparison table on the **same GPU type** (A100 results are not directly comparable with Kaggle T4 runs).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/vmtf'
DATA_ZIP   = f'{DRIVE_ROOT}/data.zip'
LOCAL_DATA = '/content/data'                       # fast local disk used for training
RESULTS    = f'{DRIVE_ROOT}/results/colab_a100'     # persistent results on Drive
TEXT_MODEL   = 'vinai/phobert-base'
VISUAL_MODEL = 'google/vit-base-patch16-224'
os.makedirs(RESULTS, exist_ok=True)

!nvidia-smi --query-gpu=name,memory.total --format=csv
print('vCPU:', os.cpu_count())

In [ ]:
%cd /content
!git clone -q https://github.com/kascik04/vietnamese-multimodal-fake-news-detection.git || (cd vietnamese-multimodal-fake-news-detection && git pull -q)
%cd /content/vietnamese-multimodal-fake-news-detection
!pip install -q pyvi imagehash
!git log --oneline -1

In [ ]:
# Copy the single zip from Drive and extract on the local disk (1-2 min, once per session)
if not os.path.exists(f'{LOCAL_DATA}/train.csv'):
    !cp "{DATA_ZIP}" /content/data.zip && unzip -q -o /content/data.zip -d {LOCAL_DATA} && rm /content/data.zip
!ls {LOCAL_DATA} | head
!ls {LOCAL_DATA}/images | wc -l

In [ ]:
# Optional: offline tests (~1-2 min)
!python test_step1.py && python test_step2.py && python test_step3.py && python test_step4.py && python test_step5.py

In [ ]:
# ---- One experiment (re-run this cell after a disconnect: it resumes automatically) ----
EXP_NAME = 'proposed_seed42'
EXP_DIR  = f'{RESULTS}/{EXP_NAME}'
BATCH    = 16   # keep the SAME batch for every model of a comparison table (32 is fine on A100 if used for all)

last = f'{EXP_DIR}/last_checkpoint.pt'
flag = f'--resume {last}' if os.path.exists(last) else ''

# auto: bf16 + TF32 on A100, fp16 on V100; workers auto-scaled to the Colab vCPU count
!python -m vmtf_net.train \
    --train_csv {LOCAL_DATA}/train.csv --val_csv {LOCAL_DATA}/val.csv --test_csv {LOCAL_DATA}/test.csv \
    --image_dir {LOCAL_DATA}/images --cache_dir /content/cache \
    --text_model_name {TEXT_MODEL} --visual_model_name {VISUAL_MODEL} \
    --fusion_type hybrid_proposed --consistency_lambda 0.1 --social_preset basic \
    --batch_size {BATCH} --epochs 10 --lr 2e-5 --head_lr 1e-4 --seed 42 \
    --checkpoint_dir {EXP_DIR} --no_tqdm {flag}

In [ ]:
# ---- Several seeds of one configuration (finished runs skipped, interrupted runs resumed) ----
!python scripts/run_seeds.py --seeds 13 42 2024 --out_dir {RESULTS}/proposed -- \
    --train_csv {LOCAL_DATA}/train.csv --val_csv {LOCAL_DATA}/val.csv --test_csv {LOCAL_DATA}/test.csv \
    --image_dir {LOCAL_DATA}/images --cache_dir /content/cache --batch_size {BATCH} --no_tqdm

In [ ]:
import json
import pandas as pd

history = pd.read_csv(f'{EXP_DIR}/history.csv')
display(history[['epoch', 'train_f1_macro', 'val_f1_macro', 'val_loss', 'epoch_seconds']])
results = json.load(open(f'{EXP_DIR}/results.json'))
print(json.dumps(results['test']['metrics'], indent=2))
print(json.dumps(json.load(open(f'{EXP_DIR}/run_info.json'))['precision'], indent=2))